# FlipArb Public Analysis Notebook

**Portfolio notebook for marketplace opportunity analysis**

This notebook walks through the business question FlipArb was built to answer and demonstrates the analytical workflow on a **synthetic** public dataset.

## Important data notice

The file `data/sample_listings.csv` is **synthetic**. It was created for public portfolio demonstration only. It does not contain production marketplace records, customer information, or private FlipArb outputs.

## Connection to the product

FlipArb scanned marketplace listings, enriched candidates with repair and device status signals, estimated profit and ROI, scored risk and confidence, and used Thompson Sampling to allocate scanning budget. This notebook focuses on the decision analytics layer using public sample data.

## 1. Business question

Which marketplace listings are worth deeper review for phone resale?

A cheap asking price is not enough. A useful opportunity should still look attractive after shipping, tax, repair cost, lock risk, comparable sale quality, liquidity, and confidence are considered.

The questions this notebook explores:

1. How do opportunities differ by sourcing strategy?
2. How do projected profit and risk interact?
3. Which listings look stronger once liquidity and confidence are included?
4. What does a practical review shortlist look like?

## 2. Setup and dataset description

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("../data/sample_listings.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nThis dataset is synthetic and for portfolio demonstration only.")
df.head()

### Field glossary

| Field | Meaning |
| --- | --- |
| `source` | Sourcing strategy that found the listing |
| `projected_profit` | Expected resale minus purchase related costs |
| `roi` | Projected return on investment |
| `confidence` | Strength of available evidence |
| `risk_level` | Coarse operational risk label |
| `comp_count` | Number of comparable sales used |
| `liquidity_score` | Relative ease of resale |
| `lock_flag` | Whether carrier or network lock risk is present |
| `bucket` | Opportunity class: alert ready, needs review, or rejected |

## 3. Exploratory analysis

In [ ]:
numeric_cols = [
    "projected_profit",
    "roi",
    "confidence",
    "comp_count",
    "liquidity_score",
    "listing_age_minutes",
]

print("Missing values:", int(df.isna().sum().sum()))
print("\nBucket mix:")
print(df["bucket"].value_counts())
print("\nRisk mix:")
print(df["risk_level"].value_counts())
df[numeric_cols].describe().round(2)

## 4. Opportunity mix by source

FlipArb did not rely on one sourcing path. Comparing sources helps show whether specialty strategies such as misspellings or ending auctions can justify part of the scanner budget.

In [ ]:
source_summary = (
    df.groupby("source")
    .agg(
        listings=("listing_id", "count"),
        avg_profit=("projected_profit", "mean"),
        avg_roi=("roi", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_liquidity=("liquidity_score", "mean"),
        alert_rate=("bucket", lambda s: (s == "alert_ready").mean()),
    )
    .sort_values("alert_rate", ascending=False)
)

source_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ordered = source_summary["alert_rate"].sort_values()
ax.barh(ordered.index, ordered.values, color="#1f5eff")
ax.set_xlabel("Alert ready rate")
ax.set_ylabel("Source")
ax.set_title("Synthetic alert ready rate by sourcing strategy")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## 5. Profitability analysis

Projected profit is necessary but not sufficient. Risk adjustment is a simple way to show that two listings with the same headline profit can deserve different priority.

In [ ]:
risk_multiplier = {"Low": 1.00, "Medium": 0.90, "High": 0.75}

df = df.copy()
df["risk_adjusted_profit"] = df.apply(
    lambda row: row["projected_profit"] * risk_multiplier[row["risk_level"]],
    axis=1,
)

profit_by_bucket = (
    df.groupby("bucket")[["projected_profit", "risk_adjusted_profit", "roi"]]
    .mean()
    .reindex(["alert_ready", "needs_review", "rejected"])
)

profit_by_bucket.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for bucket, color in [
    ("alert_ready", "#0f7a56"),
    ("needs_review", "#1f5eff"),
    ("rejected", "#c45c26"),
]:
    subset = df[df["bucket"] == bucket]
    ax.scatter(
        subset["confidence"],
        subset["projected_profit"],
        alpha=0.75,
        label=bucket,
        color=color,
    )

ax.set_xlabel("Confidence score")
ax.set_ylabel("Projected profit")
ax.set_title("Synthetic profit versus confidence by opportunity class")
ax.legend(title="Bucket")
ax.axhline(0, color="#9aa7b5", linewidth=1)
plt.tight_layout()
plt.show()

## 6. Risk analysis

High projected profit with high risk or lock exposure can still be a poor operational choice. This section inspects risk labels and lock flags against opportunity class.

In [ ]:
risk_pivot = pd.crosstab(df["risk_level"], df["bucket"], normalize="index")
lock_summary = (
    df.groupby("lock_flag")
    .agg(
        listings=("listing_id", "count"),
        avg_profit=("projected_profit", "mean"),
        avg_confidence=("confidence", "mean"),
        alert_rate=("bucket", lambda s: (s == "alert_ready").mean()),
    )
)

print("Risk level versus bucket (row percentages)")
print(risk_pivot.reindex(["Low", "Medium", "High"]).round(3))
print("\nLock flag summary")
lock_summary.round(3)


## 7. Liquidity analysis

A profitable listing that is hard to resell can tie up cash. Liquidity and comparable sale count help represent that friction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

df.boxplot(column="liquidity_score", by="bucket", ax=axes[0])
axes[0].set_title("Liquidity by opportunity class")
axes[0].set_xlabel("Bucket")
axes[0].set_ylabel("Liquidity score")

axes[1].scatter(df["comp_count"], df["liquidity_score"], alpha=0.7, color="#0e7490")
axes[1].set_xlabel("Comparable sale count")
axes[1].set_ylabel("Liquidity score")
axes[1].set_title("Liquidity versus comparable sale support")

plt.suptitle("")
plt.tight_layout()
plt.show()

(
    df.groupby("bucket")[["liquidity_score", "comp_count", "confidence"]]
    .mean()
    .reindex(["alert_ready", "needs_review", "rejected"])
    .round(2)
)

## 8. Building a simple review shortlist

The public scoring demo combines several signals. Below is a transparent shortlist rule for this synthetic dataset. It is illustrative, not a claim about production model accuracy.

In [ ]:
shortlist = df[
    (df["projected_profit"] >= 75)
    & (df["confidence"] >= 80)
    & (df["liquidity_score"] >= 55)
    & (df["risk_level"] != "High")
    & (~df["lock_flag"])
].sort_values(["risk_adjusted_profit", "confidence"], ascending=False)

print(f"Shortlist size: {len(shortlist)} of {len(df)} synthetic listings")
shortlist[
    [
        "listing_id",
        "model",
        "source",
        "projected_profit",
        "risk_adjusted_profit",
        "confidence",
        "liquidity_score",
        "bucket",
    ]
].head(10).round(2)

## 9. Conclusions

1. **Profit alone is incomplete.** Risk, confidence, liquidity, and lock status change which listings deserve attention.
2. **Source mix matters.** Different sourcing strategies can produce different opportunity quality profiles, which is why FlipArb allocated scanner budget across multiple paths and learned which queries performed better.
3. **Evidence quality is part of the decision.** Comparable sale support and confidence help separate fragile estimates from stronger ones.
4. **This notebook is a portfolio lens, not a production audit.** The dataset is synthetic. Measured FlipArb scale metrics such as the recorded Deal Engine session belong to the product documentation and website, not to this sample file.

## Next steps for reviewers

* Portfolio site: `docs/index.html`
* Scoring demo: `src/scoring_demo.py`
* Thompson Sampling demo: `src/thompson_sampling_demo.py`
* Learning system notes: `docs/LEARNING_SYSTEM.md`